# 01. Getting the permit data

**COMM 1280 Final Data Project** · Jesus Aguirre and Lisandro Collazo

This notebook downloads the data and saves it in `../data/raw/`.
It does not change anything. Cleaning happens in notebook 03.

## Where the data comes from

Philadelphia publishes every building permit it issues on OpenDataPhilly.
We ask their server for the rows we want instead of downloading one huge file.

One row is one permit. Each row has the date the permit was issued, the
address, the census tract, and what kind of work it is for.

In [ ]:
import pandas as pd
import requests
import os

API = 'https://phl.carto.com/api/v2/sql'
RAW = '../data/raw/'

os.makedirs(RAW, exist_ok=True)

In [ ]:
def download(sql, filename):
    # Send the question to the city's server and save the answer as a CSV.
    response = requests.get(API, params={'q': sql, 'format': 'csv'})
    outfile = open(RAW + filename, 'w', encoding='utf-8')
    outfile.write(response.text)
    outfile.close()
    print('saved', filename)

## Which permits should we count?

This is the most important choice in the project.

One construction project does not create one permit. It creates several. The
builder gets a building permit, and then separate permits for electrical work,
plumbing, heating and fire sprinklers. If we counted all of them, one house
would show up four or five times.

Let's look at what is actually in the data before we decide.

In [ ]:
sql = """
SELECT permitdescription, count(*) AS number_of_permits
FROM permits
WHERE typeofwork = 'New Construction'
GROUP BY permitdescription
ORDER BY number_of_permits DESC
"""

response = requests.get(API, params={'q': sql})
pd.DataFrame(response.json()['rows'])

The electrical, mechanical and fire sprinkler permits are the most common
rows, just as we expected. They belong to buildings that already have a
building permit.

So we keep only two kinds:

- `Residential Building Permit`, the homes we are studying
- `Commercial Building Permit`, which we planned to use for comparison

We also keep only `New Construction`, because remodels never qualified for
the tax break.

In [ ]:
sql = """
SELECT permitnumber, permitissuedate, permitdescription, address, zip,
       censustract, council_district, opa_account_num, approvedscopeofwork
FROM permits
WHERE typeofwork = 'New Construction'
  AND permitdescription IN ('Residential Building Permit', 'Commercial Building Permit')
  AND permitissuedate >= '2016-01-01'
  AND permitissuedate < '2025-01-01'
ORDER BY permitissuedate
"""

download(sql, 'new_construction_building_permits_2016_2024.csv')

In [ ]:
permits = pd.read_csv(RAW + 'new_construction_building_permits_2016_2024.csv')

print('number of permits:', len(permits))
permits.head()

## Two more files for the data check

Notebook 02 asks whether the older years can be trusted. To answer that we
need two more small files.

The first counts **every** permit per year, not just new construction. If the
total stays flat while new construction seems to explode, then the explosion
is in the paperwork and not in the city.

The second splits permits by which computer system recorded them. The old
system wrote categories in CAPITAL LETTERS and the new one does not, so we can
count each style and see when the switch finished.

In [ ]:
sql = """
SELECT extract(year from permitissuedate) AS year,
       count(*) AS all_permits,
       sum(case when typeofwork = 'New Construction' then 1 else 0 end) AS new_construction
FROM permits
WHERE permitissuedate >= '2016-01-01' AND permitissuedate < '2025-01-01'
GROUP BY year
ORDER BY year
"""

download(sql, 'all_permits_by_year_2016_2024.csv')
pd.read_csv(RAW + 'all_permits_by_year_2016_2024.csv')

In [ ]:
sql = """
SELECT to_char(permitissuedate, 'YYYY-MM') AS month,
       sum(case when permitdescription = upper(permitdescription) then 1 else 0 end) AS old_system,
       sum(case when permitdescription <> upper(permitdescription) then 1 else 0 end) AS new_system
FROM permits
WHERE permitissuedate >= '2016-01-01' AND permitissuedate < '2025-01-01'
GROUP BY month
ORDER BY month
"""

download(sql, 'permit_record_system_by_month_2016_2024.csv')
pd.read_csv(RAW + 'permit_record_system_by_month_2016_2024.csv').tail()

## What we saved

| File | What it holds |
| --- | --- |
| `new_construction_building_permits_2016_2024.csv` | One row per new construction building permit |
| `all_permits_by_year_2016_2024.csv` | Every permit counted by year |
| `permit_record_system_by_month_2016_2024.csv` | Old system against new system, by month |

Next: `02_data_quality_and_coverage.ipynb`.